In [1]:
%pip install --upgrade google-adk google-cloud-aiplatform litellm requests ipdb google-cloud-modelarmor --quiet

In [3]:
import os
from typing import Dict, List, Optional
from IPython.display import display, Markdown
import requests
import vertexai
from google.adk.agents import Agent
from google.adk import Workflow
from google.adk.models.lite_llm import LiteLlm
from google.adk.models import LlmResponse, LlmRequest
from google.adk.agents.callback_context import CallbackContext
from vertexai.preview import reasoning_engines
from google.adk.tools import google_search
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("safetydance_agent")

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

# Pull secrets and config from the environment — never hardcode these.
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")  # required by LiteLLM for Claude
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")

for name, value in [
    ("GOOGLE_MAPS_API_KEY", GOOGLE_MAPS_API_KEY),
    ("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY),
    ("GOOGLE_CLOUD_PROJECT", PROJECT_ID),
]:
    if not value:
        print(f"WARNING: {name} is not set in the environment.")

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [4]:
def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """
    Convert a place name or address into latitude and longitude using the
    Google Maps Geocoding API.

    Args:
        location (str): A place name, city, or address (e.g., "College Station, TX").

    Returns:
        Optional[Dict[str, float]]: A dictionary with 'lat' and 'lon' keys.
        Returns None if the location cannot be found or an error occurs.
    """
    # breakpoint()
    if not GOOGLE_MAPS_API_KEY:
        return "google api key not found"

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        coords = data["results"][0]["geometry"]["location"]
        return {"lat": coords["lat"], "lon": coords["lng"]}
    except (requests.RequestException, KeyError, IndexError):
        return "exception, try again"

In [5]:
print(get_lat_lon("College Station, TX"))

{'lat': 30.6210482, 'lon': -96.3255016}


In [6]:
def get_gov_weather_forecast(
    lat: float, lon: float
) -> Optional[List[Dict[str, str]]]:
    """
    Fetch today's weather forecast from the U.S. National Weather Service
    API based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each with 'name', 'temperature', 'shortForecast', and 'detailedForecast'.
        Returns None if data is unavailable (including for non-US locations,
        which NWS does not cover) or an error occurs.
    """
    # NWS requires a descriptive User-Agent identifying the application.
    headers = {"User-Agent": "(google-lab-readynow-weather-agent)"}

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": period["name"],
                "temperature": f"{period['temperature']}°{period['temperatureUnit']}",
                "shortForecast": period["shortForecast"],
                # "detailedForecast": period["detailedForecast"],
            }
            for period in periods
        ]
    except (requests.RequestException, KeyError):
        return None

In [7]:
WEATHER_AGENT_INSTRUCTIONS = """
You are a helpful weather assistant covering locations in the United States.

When a user asks about weather for a location:
1. Call get_lat_lon to convert the location name into latitude/longitude.
2. Call get_gov_weather_forecast with those coordinates to retrieve
   the forecast from the National Weather Service.
3. Summarize the forecast in plain, easy-to-read language. If any period's
   forecast mentions severe weather (storms, extreme heat, flooding, winter
   weather, etc.), lead your response with a clear "ALERT:" line.

If get_lat_lon returns None, tell the user you couldn't find that location.
If get_gov_weather_forecast returns None, tell the user the forecast
is unavailable, and note that the National Weather Service only covers
US locations.
"""

In [8]:

from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

_MODEL_ARMOR_LOCATION = os.getenv("MODEL_ARMOR_LOCATION", "us-central1")
_MODEL_ARMOR_TEMPLATE = os.getenv("MODEL_ARMOR_TEMPLATE", "safety-dance")

_model_armor_client = modelarmor_v1.ModelArmorClient(
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{_MODEL_ARMOR_LOCATION}.rep.googleapis.com"
    )
)
_model_armor_template_name = (
    f"projects/{PROJECT_ID}/locations/{_MODEL_ARMOR_LOCATION}/templates/{_MODEL_ARMOR_TEMPLATE}"
)


def check_user_input(user_input: str) -> str:
    """
    Screen user input for harmful, abusive, or malicious content using
    Google Cloud Model Armor.

    Args:
        user_input (str): The raw text a user submitted to the agent.

    Returns:
        str: "BAD" if Model Armor flags the input, "OK" otherwise. Fails
        open (returns "OK") if the Model Armor call itself errors, so a
        service outage doesn't block legitimate users.
    """
    try:
        response = _model_armor_client.sanitize_user_prompt(
            request=modelarmor_v1.SanitizeUserPromptRequest(
                name=_model_armor_template_name,
                user_prompt_data=modelarmor_v1.DataItem(text=user_input),
            )
        )
        match_state = response.sanitization_result.filter_match_state
        return "BAD" if match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND else "OK"
    except Exception:
        return "OK"

In [9]:
def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """
    Log the most recent user message before it is sent to the model.

    Args:
        callback_context (CallbackContext): Context for the current agent
            invocation, including the agent's name.
        llm_request (LlmRequest): The request about to be sent to the model.

    Returns:
        Optional[LlmResponse]: Always returns None, allowing processing
        to continue.
    """
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            logger.info(
                "[%s] USER » %s", callback_context.agent_name, last.parts[0].text.strip()
            )

    return None

In [10]:
def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """
    Log the model's response after it is generated, before it is returned
    to the user.

    Args:
        callback_context (CallbackContext): Context for the current agent
            invocation, including the agent's name.
        llm_response (LlmResponse): The response generated by the model.

    Returns:
        Optional[LlmResponse]: Always returns None, allowing processing
        to continue.
    """
    if llm_response.content and llm_response.content.parts:
        text = llm_response.content.parts[0].text
        if text:
            logger.info("[%s] MODEL » %s", callback_context.agent_name, text.strip())

    return None

In [11]:
def moderate_user_prompt(callback_context: CallbackContext,llm_request: LlmRequest
) -> Optional[LlmResponse]:
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if last.role != "user" or not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()
        result_text = check_user_input(user_text)

        if result_text.strip().upper() == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "⚠️ Sorry, that message violates our content guidelines."}]
            })

    except Exception as e:
        import logging
        logging.exception("Moderation callback failed: %s", e)

    return None  # Proceed with model call


In [12]:
def chained_before_callback(callback_context, llm_request):
# 1. Moderation check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if  moderation_result is not None:
    return moderation_result  # STOP: message was inappropriate
# 2. Log user input (optional)
  log_user_prompt(callback_context, llm_request)
  return None
# Allow agent to proceed

In [13]:
def search_for_fun(location: str, weather: str) -> str:
  """
  Find fun things to do in a given location based on the current weather.

  Args:
      location (string): College Station, TX
      weather (string): Most days will see temperatures in the high 90s to low 100s, with heat index values reaching as high as 105°F early in the week.

  Returns:
      Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
      each with 'name', 'temperature', 'shortForecast', and 'detailedForecast'.
      Returns None if data is unavailable (including for non-US locations,
      which NWS does not cover) or an error occurs.
  """
  query = f"fun things to do in {location} when the weather outside is {weather}"
  google_search_result = google_search.run(query)
  print(type(google_search_result))
  return google_search_result

In [14]:
weather_agent_with_moderation = Agent(
   name="marvin",
   model="gemini-2.5-flash",
    instruction="Get today's weather for {location}.",
    tools=[get_lat_lon, get_gov_weather_forecast],
    output_key="weather_today",
)
validator = Agent(
    name="ValidateInput",
    model="gemini-2.5-flash",
    instruction="Ensure the user has provided a valid location. Respond with just the location name.",
    output_key="location",
)

fun_finder = Agent(
    name="FunFinder",
    model="gemini-2.5-flash",
    instruction="Find fun things to do today based on {weather_today}.",
    output_key="result",
    tools=[google_search],
)

In [15]:
from google.adk.agents import SequentialAgent

def build_app(agent: Agent) -> reasoning_engines.AdkApp:
    """Wrap an ADK agent in an AdkApp so it can be run locally."""
    return reasoning_engines.AdkApp(agent=agent)

def ask_agent(app: reasoning_engines.AdkApp, user_id: str, session_id: str, message: str) -> str:
    """
    Send a message to an AdkApp for a given session and return the final
    text response.

    Args:
        app: The AdkApp instance to query.
        user_id: The user ID associated with the session.
        session_id: The session ID to send the message within.
        message: The user's message to the agent.

    Returns:
        str: The agent's final text response, or a fallback message if
        no usable response was returned.
    """
    last_event = None
    for event in app.stream_query(user_id=user_id, session_id=session_id, message=message):
        last_event = event

    try:
        return last_event["content"]["parts"][0]["text"]
    except (TypeError, KeyError, IndexError):
        return "No response received from the agent."

root_agent = SequentialAgent(
    name="BoredomAbliterator",
    description="Find fun things to do in a given location based on the current weather.",
    sub_agents=[validator, weather_agent_with_moderation, fun_finder],
)


/tmp/ipykernel_104577/1157567885.py:31: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [16]:
# boredom_app = build_app(root_agent)
# boredom_user_id = "test-user-id"
# boredom_session = boredom_app.create_session(user_id=boredom_user_id)

# final_result = ask_agent(
#     boredom_app,
#     boredom_user_id,
#     boredom_session['id'],
#     "I'm in College Station, TX.",
# )

# print(final_result)

In [17]:
boredom_app = build_app(root_agent)
boredom_user_id = "test-user-id"
you_are_here = input("Where are you today? ")
while you_are_here != "quit":
    boredom_session = boredom_app.create_session(user_id=boredom_user_id)
    display(Markdown(f"## Fun things to do in {you_are_here}"))
    display(Markdown(ask_agent(boredom_app, boredom_user_id, boredom_session['id'], you_are_here)))
    you_are_here = input("Where are you today? ")


Where are you today? Boston, MA


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


## Fun things to do in Boston, MA

/usr/local/lib/python3.12/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


Tonight in Boston, with a mostly clear sky and a comfortable 65°F, offers a variety of enjoyable options for your Monday evening. Since it's currently 7:51 PM EDT, many places are still open or just getting started.

Here are some fun things to do:

**For Late-Night Bites and Drinks:**

*   **Citizen Public House and Oyster Bar** is a cozy English-style pub open until 1 AM on Mondays, offering a comprehensive whiskey menu and tavern food, including a raw bar.
*   If you're in the mood for Mexican, **Lone Star Taco Bar** has locations in Allston and Cambridge, with the Allston spot open until 1 AM daily and the Cambridge one until 12 AM. They offer tacos, chips with salsa, and unique to-go cocktails.
*   For classic American comfort food, **Jim Curley** near Downtown Crossing serves award-winning burgers and brews until 2 AM every night.
*   **Bova's Bakery** in the North End is open 24 hours a day, seven days a week, offering a selection of baked goods, including cannolis, turnovers, cream puffs, and even homemade pizza and calzones.
*   **El Jefe's Taqueria** has several locations and is known for burritos, open until 2 AM.
*   **South Street Diner** in Chinatown is another 24/7 establishment, serving breakfast and dinner.
*   **Anatolia Kebab House** offers Turkish and American fusion food, including kebabs, pasta, and pizza, until 1:30 AM.
*   **Russell House Tavern** offers a late-night menu until 11:00 PM on Mondays.
*   **The Rail Trail Flatbread Co.** has a tavern menu available until 12:00 AM on Mondays.

**For Nightlife and Entertainment:**

*   Enjoy live jazz at **Wally's Cafe**, a Roxbury jazz dive that has been featuring live jazz nightly since 1947 and is open until 2 AM. Monday nights are specifically blues nights.
*   **A4Cade** in Cambridge and **Versus** near Downtown Crossing are arcade bars where you can play old-school coin-op arcade games and pinball while enjoying drinks. Versus is open from 5 PM on Mondays, and A4Cade is also a good option.
*   The **Encore Casino** in Everett is open 24/7, with booze served until 4 AM. It offers an opportunity for people-watching and gaming.
*   **Monday Movie Nights at the Market** at Time Out Market Boston are recurring weekly until September 21, 2026, where the lawn at 401 Park transforms into an outdoor cinema.
*   You could also check out **Alibi Bar and Lounge** in the Liberty Hotel, which is a trendy bar and often has a packed patio when the weather is warm.
*   **Distraction + Democracy Beer Garden** at City Hall Plaza is open until 8 PM on Mondays, offering locally made beer, wine, and seltzers, along with live music.

Considering the pleasant weather, exploring some of the outdoor bar options like **The Landing at Long Wharf** or **Rowes Wharf Sea Grille** for a drink with a view could also be enjoyable, though patio closing times might vary.

Before heading out, it's always a good idea to check the specific operating hours of any venue as they can change.

...

Here are some fun things to do in Houston, TX, based on the weather forecast:

**Tonight: Partly Cloudy with a temperature of 80°F**
With pleasant evening temperatures, you have a variety of options for indoor and outdoor activities:
*   **Nightlife and Live Music** Houston boasts a vibrant nightlife, even on a Monday. Consider visiting places like AvantGarden for experimental sounds, McGonigel's Mucky Duck for an open mic night, or Urban Smoke and Urban Social for a DJ and club atmosphere. La Carafe, a historic downtown bar, offers a unique experience, while Grand Prize Bar features drag shows and events. Social Beer Garden is open until 8:00 PM, offering craft beer, cocktails, and food.
*   **Outdoor Stroll** Enjoy the comfortable evening weather with a night stroll in Buffalo Bayou Park, which has paved, well-lit walkways and is open 24 hours. Discovery Green also often features public art installations, such as the Art of Light, which can be enjoyed in the evening.
*   **Indoor Entertainment** If you prefer to stay indoors, Houston offers several exciting options. You can challenge yourself at one of the city's escape rooms, many of which offer late-night sessions. Sing your heart out at Spotlight Karaoke. Explore immersive art at Meow Wolf Houston or ARTECHOUSE Houston, both typically open until 8:00 PM on weekdays. Palace Social provides a comprehensive entertainment complex with boutique bowling, VR gaming, arcade games, and karaoke.
*   **Theater Week** Houston Theater Week is currently running from August 24-30, offering buy-one-get-one-free deals on tickets to various performing arts shows.

**Tuesday: Partly Sunny with a temperature of 98°F**
Given the high temperatures, indoor and water-based activities are highly recommended to beat the heat:
*   **Museum Exploration** Houston's Museum District is an excellent way to spend a hot day in air-conditioned comfort. The Houston Museum of Natural Science offers free admission on Tuesdays from 5:00 PM to 8:00 PM. Other notable museums include the Children's Museum Houston, the Museum of Fine Arts, Houston (MFAH), the Health Museum, and the always-free Contemporary Arts Museum Houston and The Menil Collection. You could also visit the Museum of Illusions Houston or the Space Center Houston.
*   **Indoor Recreation and Entertainment** Escape the heat by walking through Houston's underground Downtown Tunnels, which are climate-controlled and feature shops and eateries. Catch a movie at one of Houston's many theaters, including iPic Cinema for a luxury experience or the Houston Museum of Natural Science's Giant Screen Theater for 3D documentaries. For active fun, consider ice skating at Ice Skate USA in Memorial City Mall or Ice at the Galleria. Other indoor options include Cidercade arcade with over 275 classic games, Claw & Fun in Katy, trampoline parks like Jumping World, or Battlefield Houston for tactical laser tag. The Houston Public Library offers a cool and quiet retreat. You can also find indoor activities like axe throwing, paintball, and laser tag at places like AGR Sports Club.
*   **Cool Off in Water** If you prefer to brave the outdoors and cool off, many public pools and splash pads operated by the Houston Parks & Recreation Department are open from 1:00 PM to 8:00 PM on Tuesdays and offer free water fitness classes. Noah's Ark Pool in West Houston is another family-friendly option. You could also visit Discovery Green's Gateway Fountain for free water fun, or consider a day pass to a hotel pool featuring refreshing amenities like the Texas-shaped lazy river at the Marriott Marquis Houston. Water parks such as Typhoon Texas in Katy also provide an excellent way to beat the heat.

Where are you today? cardboard box


## Fun things to do in cardboard box

That doesn't look like a valid location. Could you please provide a real place name? I need a valid location (like a city, state, or even a specific address) to help you find fun things to do.

Where are you today? quit
